[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/icosahedron.ipynb)

# Icosahedral Symmetry from Three Mirrors

This example constructs the full symmetry group of the icosahedron from three mirrors meeting at the centre of a sphere. Three planes meeting at angles of a half turn divided by two, three and five cut out a spherical triangle, and repeated reflections across those planes partition the sphere into 120 congruent triangles, half reached by pure rotations and half as mirror images. Pairwise intersections of the mirrors form the twofold, threefold and fivefold symmetry axes on which the edge midpoints, face centres and vertices of the icosahedron lie.

Each mirror is modelled as an extrinsic unit plane `Plane` and each point of the sphere as an intrinsic point `Point`, so reflecting across a mirror is the sandwich `mirror >> points`. Multiplying two mirror planes such as `third * second` forms a rotor about their line of intersection that turns by twice the dihedral angle between them. The third mirror follows directly from its inner products with the first two via the reciprocal vectors `second | plane.inverse()` and `plane.inverse() | first`, while the regressive pairing `mirror & points` tests which side of each mirror a point lies on and counts the reflections needed to fold it into the fundamental triangle.

In matrix Householder and Coxeter notation, reflections in unit normals $\mathbf{n}_k$ act on column vectors via $H_k = I - 2\,\mathbf{n}_k\mathbf{n}_k^{\mathsf T}$ with $\det H_k = -1$, pairwise rotations require matrix products $R_{jk} = H_j H_k$ satisfying $(H_1 H_2)^2 = (H_1 H_3)^3 = (H_2 H_3)^5 = I$ in the group $[5, 3]$, and orientation parity is tracked separately through $\det(H_{k_m}\cdots H_{k_1}) = (-1)^m$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import show_states
from examples.math.reflections import render

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
Plane = ga.gatype.vector()                                                      # Plane+
Point = ga.gatype.antivector()                                                  # Point-

## Mirror planes and dihedral angles

Outward-pointing unit mirror vectors have negative inner products equal to minus the cosine of the dihedral angle between their planes. Placing the first two mirrors in the horizontal plane fixes the third mirror's horizontal component from its two prescribed inner products via the plane's reciprocal basis, while unit normalization fixes its vertical component.

In [ ]:
# Dihedral angles between pairs of mirrors.
first_second = np.pi / 2
first_third = np.pi / 3
second_third = np.pi / 5

In [ ]:
first = mv.x                                                                    # [] Plane+
first_turn = (mv.xy * (-(np.pi - first_second) / 2)).exp()                      # [] Rotor+
second = first_turn >> first                                                    # [] Plane+
# Reciprocal vectors in `plane` isolate each prescribed inner product with `first` and `second`.
plane = first ^ second                                                          # [] Bivector+
in_plane = -np.cos(first_third) * (second | plane.inverse()) - np.cos(second_third) * (plane.inverse() | first)   # [] Plane+
normal = mv.z                                                                   # [] Plane+
height = (1 - in_plane.squared()).square_root()                                 # [] Scalar+
third = in_plane + height * normal                                              # [] Plane+
mirrors = stack([first, second, third])                                         # [3] Plane+

# Pairwise products of mirrors form rotors around their intersection axes.
twofold = second * first                                                        # [] Rotor+
threefold = third * first                                                       # [] Rotor+
fivefold = third * second                                                       # [] Rotor+

In [ ]:
print("pairings of the mirror planes:")
print((mirrors[:, None] | mirrors[None, :]).to_array())

## Folding the sphere into the fundamental triangle

Wherever `mirror & points` is negative, a point lies outside that mirror's half-space and reflects across it. Sweeping through the three mirrors over five rounds folds every point of the sphere into the fundamental triangle and counts how many reflections it underwent, separating direct rotational copies of even parity from reflected copies of odd parity.

In [ ]:
# Pixel resolution across the disk and reflection sweeps needed to fold the whole sphere.
n_pixels = 240
n_rounds = 5

In [ ]:
def folded_sphere(points: Point, mirrors: Plane, n_rounds: int) -> tuple[Point, Scalar]:
    """Repeated reflections across `mirrors` fold `points` (`[...] Point-`) into the fundamental triangle and count the flips (`[...] Scalar+`)."""
    flip_counts = mv.scalar(np.zeros(points.shape + (1,)))                      # [...] Scalar+
    for _ in range(n_rounds):
        for index in range(mirrors.shape[0]):
            side = mirrors[index] & points                                      # [...] Scalar+
            outside = (side.abs() - side) / (2 * side.abs())                    # [...] Scalar+
            points = points + outside * ((mirrors[index] >> points) - points)
            flip_counts = flip_counts + outside
    return points, flip_counts


# Orthographic projection of the visible hemisphere onto the pixel grid.
horizontal, vertical = render.disk(n_pixels)
disk_points = mv.x.dual() * horizontal + mv.y.dual() * vertical                 # [n_pixels, n_pixels] Point-
screen_points = disk_points + mv.z.dual() * (1 - disk_points.scalar_norm_squared()).clip(0.0, None).square_root()   # [n_pixels, n_pixels] Point-

## Turning the tiled icosahedron

Turning the tilted sphere through a half turn around the twofold axis where the first two mirrors meet carries the tiling onto itself.

In [ ]:
# Frame count, frame duration, and camera tilt angle.
n_frames = 108
frame_duration_ms = 40
tilt = 0.4

In [ ]:
def sphere_frames(n_frames: int) -> Iterator[tuple[Point, Scalar]]:
    """Yields `(folded_points, flip_counts)` across `n_frames` steps of a half turn of the tilted sphere."""
    tilt_rotor = (mv.yz * (-tilt / 2)).exp()                                    # [] Rotor+
    tilted_points = tilt_rotor >> screen_points                                 # [n_pixels, n_pixels] Point-
    for angle in np.linspace(0.0, np.pi, n_frames, endpoint=False):
        turn_rotor = (plane * (-angle / 2)).exp()                               # [] Rotor+
        turned_points = turn_rotor >> tilted_points                             # [n_pixels, n_pixels] Point-
        yield folded_sphere(turned_points, mirrors, n_rounds)


recorded_frames = show_states(
    sphere_frames(n_frames), render.sphere_animation, screen_points, mirrors, second,
    name="icosahedron", duration_ms=frame_duration_ms, n_frames=n_frames,
);

Warm gold and slate blue tiles alternate between even and odd reflection counts across the 120 copies of the fundamental triangle, thin dark arcs mark all three mirror planes, and bold arcs along the second mirror outline the twenty triangular faces of the icosahedron.

In [ ]:
# checks
for one, other, angle in ((first, second, first_second), (first, third, first_third), (second, third, second_third)):
    assert (((one | other) + np.cos(angle)).abs() < 1e-10).all()
assert (((mirrors | mirrors) - 1).abs() < 1e-10).all()
for rotor, order in ((twofold, 2), (threefold, 3), (fivefold, 5)):
    power = rotor                                                               # [] Rotor+
    for _ in range(order - 1):
        power = power * rotor                                                   # [] Rotor+
    assert ((power.select_grade(0).abs() - 1).abs() < 1e-10).all()
    assert (power.select_grade(2).norm() < 1e-10).all()
first_folded, first_flips = recorded_frames[0]
inside_disk = horizontal**2 + vertical**2 < 1
assert ((mirrors & first_folded[..., None]) > -1e-10)[inside_disk].all()